# Run Multi-Model Evaluations - Complete Workflow

This notebook runs the complete evaluation workflow:
1. **Load configuration** from .env file
2. **Upload validation dataset** to Azure AI Foundry
3. **Create evaluation definition** with custom Python grader
4. **Launch eval runs** for all 6 deployed agents
5. **Monitor progress** and wait for completion
6. **Display results** with comparative analysis

**Prerequisites:**
- Copy `.env.example` to `.env` and fill in your `PROJECT_ENDPOINT`
- Agents deployed (see `02_deploy_agents.md`)
- Custom Python grader (`eval/zava_grader_response.py`)
- Validation data (`data/rft_next_val.jsonl`)

## 1. Setup and Configuration

Load project configuration from `.env` file.

In [1]:
import os
import json
import requests
import time
from datetime import datetime
from pathlib import Path
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get project configuration from environment
PROJECT_ENDPOINT = os.getenv('PROJECT_ENDPOINT')
if not PROJECT_ENDPOINT:
    raise ValueError(
        "PROJECT_ENDPOINT not found in .env file.\n"
        "Please copy .env.example to .env and fill in your project endpoint."
    )

# Parse account and project from endpoint
# Format: https://{account}.services.ai.azure.com/api/projects/{project}
parts = PROJECT_ENDPOINT.split('/')
ACCOUNT = parts[2].split('.')[0]
PROJECT = parts[-1]
BASE_URL = f"{PROJECT_ENDPOINT}/openai"

# New dataset name (change for each run)
TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
DATASET_NAME = f"zava-eval-{TIMESTAMP}"
EVAL_NAME = f"zava-multi-model-eval-{TIMESTAMP}"

# Get credential
credential = DefaultAzureCredential()

def get_headers():
    """Get authorization headers for API requests"""
    token = credential.get_token("https://ai.azure.com/.default").token
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

print(f"✅ Configuration loaded from .env")
print(f"✅ Project Endpoint: {PROJECT_ENDPOINT}")
print(f"✅ Account: {ACCOUNT}")
print(f"✅ Project: {PROJECT}")
print(f"\n📊 Evaluation Run:")
print(f"   Dataset: {DATASET_NAME}")
print(f"   Eval: {EVAL_NAME}")

✅ Configuration loaded from .env
✅ Project Endpoint: https://ai-account-44mf5lkxqssxm.services.ai.azure.com/api/projects/ai-project-omi-build26-azd-env
✅ Account: ai-account-44mf5lkxqssxm
✅ Project: ai-project-omi-build26-azd-env

📊 Evaluation Run:
   Dataset: zava-eval-20260526-210056
   Eval: zava-multi-model-eval-20260526-210056


## 2. Upload Validation Dataset

Upload `rft_next_val.jsonl` to Azure AI Foundry using the SDK.

In [2]:
# Initialize AI Project Client
client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential
)

# Upload validation dataset
print("Uploading validation dataset...")
dataset = client.datasets.upload_file(
    file_path="../data/rft_next_val_v2.jsonl",
    name=DATASET_NAME,
    version="1.0"
)

print(f"✅ Dataset uploaded: {dataset.name}")
print(f"   ID: {dataset.id}")
print(f"   Version: {dataset.version}")

# Construct dataset URI
DATASET_URI = f"azureai://accounts/{ACCOUNT}/projects/{PROJECT}/data/{dataset.name}/versions/{dataset.version}"
print(f"   URI: {DATASET_URI}")

Uploading validation dataset...
✅ Dataset uploaded: zava-eval-20260526-210056
   ID: azureai://accounts/ai-account-44mf5lkxqssxm/projects/ai-project-omi-build26-azd-env/data/zava-eval-20260526-210056/versions/1.0
   Version: 1.0
   URI: azureai://accounts/ai-account-44mf5lkxqssxm/projects/ai-project-omi-build26-azd-env/data/zava-eval-20260526-210056/versions/1.0


## 3. Load Custom Grader

Our response-only grader scores on 3 dimensions:
- **Decision Correctness (50%):** Did the agent take the right action?
- **Financial Accuracy (30%):** Are dollar amounts correct?
- **Format Compliance (20%):** Does output follow the required format?

In [4]:
# Read the grader source code
with open('../eval/zava_grader_response.py', 'r') as f:
    GRADER_SOURCE = f.read()

print(f"✅ Loaded grader ({len(GRADER_SOURCE)} chars)")
print(f"\nGrader scoring:")
print(f"  • Decision Correctness: 50%")
print(f"  • Financial Accuracy:   30%")
print(f"  • Format Compliance:    20%")
print(f"\nPass threshold: 80% (0.8)")

✅ Loaded grader (20776 chars)

Grader scoring:
  • Decision Correctness: 50%
  • Financial Accuracy:   30%
  • Format Compliance:    20%

Pass threshold: 80% (0.8)


## 4. Create Evaluation Definition

Creates an eval with:
- Custom Python grader (`zava_quality`)
- Built-in `IntentResolution` evaluator
- Built-in `TaskCompletion` evaluator

In [5]:
eval_body = {
    "name": EVAL_NAME,
    "data_source_config": {
        "type": "custom",
        "include_sample_schema": True,
        "item_schema": {
            "type": "object",
            "properties": {
                "messages": {"type": "array"},
                "expected_resolution": {"type": "string"},
                "expected_actions": {"type": "object"},
                "expected_amounts": {"type": "object"}
            }
        }
    },
    "testing_criteria": [
        {
            "name": "zava_quality",
            "type": "python",
            "source": GRADER_SOURCE,
            "pass_threshold": 0.8
        },
        {
            "type": "azure_ai_evaluator",
            "name": "IntentResolution",
            "evaluator_name": "builtin.intent_resolution",
            "initialization_parameters": {"deployment_name": "gpt-5-4"},
            "data_mapping": {
                "query": "{{item.messages}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.expected_resolution}}"
            }
        },
        {
            "type": "azure_ai_evaluator",
            "name": "TaskCompletion",
            "evaluator_name": "builtin.task_completion",
            "initialization_parameters": {"deployment_name": "gpt-5-4"},
            "data_mapping": {
                "query": "{{item.messages}}",
                "response": "{{sample.output_text}}",
                "ground_truth": "{{item.expected_resolution}}"
            }
        }
    ]
}

url = f"{BASE_URL}/evals?api-version=2025-11-15-preview"
response = requests.post(url, headers=get_headers(), json=eval_body)

if response.status_code in [200, 201]:
    eval_data = response.json()
    eval_id = eval_data["id"]
    print(f"✅ Evaluation created: {eval_id}")
    print(f"   Name: {eval_data.get('name')}")
    print(f"   Criteria: {len(eval_body['testing_criteria'])}")
else:
    print(f"❌ Failed: {response.status_code}")
    print(response.text)
    raise Exception("Failed to create evaluation")

✅ Evaluation created: eval_09201956538e459b8773c3502b7f257b
   Name: zava-multi-model-eval-20260526-210056
   Criteria: 3


## 5. Launch Eval Runs for All Models

Running evaluations against 6 model variants:
- o4-mini
- gpt-4.1, gpt-4.1-mini, gpt-4.1-nano  
- gpt-5.4, gpt-5.4-mini

In [6]:
# All deployed agents
AGENTS = {
    "o4-mini": "zava-next-o4-mini",
    "gpt-4-1": "zava-next-gpt-4-1",
    "gpt-4-1-mini": "zava-next-gpt-4-1-mini",
    "gpt-4-1-nano": "zava-next-gpt-4-1-nano",
    "gpt-5-4": "zava-next-gpt-5-4",
    "gpt-5-4-mini": "zava-next-gpt-5-4-mini"
}

run_ids = {}
print(f"Launching eval runs against dataset: {DATASET_URI}\n")

for model_name, agent_name in AGENTS.items():
    run_config = {
        "name": f"eval-{model_name}-{TIMESTAMP}",
        "data_source": {
            "type": "azure_ai_target_completions",
            "source": {"type": "file_id", "id": DATASET_URI},
            "target": {"type": "azure_ai_agent", "name": agent_name},
            "input_messages": {"type": "item_reference", "item_reference": "item.messages"}
        }
    }
    
    url = f"{BASE_URL}/evals/{eval_id}/runs?api-version=2025-11-15-preview"
    response = requests.post(url, headers=get_headers(), json=run_config)
    
    if response.status_code in [200, 201]:
        run_ids[model_name] = response.json()["id"]
        print(f"✅ {model_name:15s} → {run_ids[model_name]}")
    else:
        print(f"❌ {model_name:15s} → ERROR {response.status_code}")
        print(f"   {response.text[:200]}")

print(f"\n🚀 Launched {len(run_ids)} eval runs")
print(f"⏱️  Expected completion time: 15-20 minutes")

Launching eval runs against dataset: azureai://accounts/ai-account-44mf5lkxqssxm/projects/ai-project-omi-build26-azd-env/data/zava-eval-20260526-210056/versions/1.0

✅ o4-mini         → evalrun_b73b33f0c03640c9ac6377aa7cb7192f
✅ gpt-4-1         → evalrun_c3192f58bf6d43c8979c6c8e05e8a281
✅ gpt-4-1-mini    → evalrun_083b49745707493f90c3ed5272235c4b
✅ gpt-4-1-nano    → evalrun_7ebb4fcea13446b4954f0ded72f990e7
✅ gpt-5-4         → evalrun_2e7349902eae416eaa61767db5bbc3b4
✅ gpt-5-4-mini    → evalrun_a020267b398f4802a85e331113d0addc

🚀 Launched 6 eval runs
⏱️  Expected completion time: 15-20 minutes


## 6. Monitor Progress

Polls eval runs every 30 seconds until all complete.

In [7]:
def check_status():
    """Check status of all runs, return counts by status"""
    status_counts = {'completed': 0, 'in_progress': 0, 'failed': 0}
    
    for model_name, run_id in run_ids.items():
        url = f"{BASE_URL}/evals/{eval_id}/runs/{run_id}?api-version=2025-11-15-preview"
        response = requests.get(url, headers=get_headers())
        
        if response.status_code == 200:
            status = response.json().get('status')
            if status in ['completed', 'succeeded']:
                status_counts['completed'] += 1
            elif status == 'failed':
                status_counts['failed'] += 1
            else:
                status_counts['in_progress'] += 1
    
    return status_counts

# Monitor until all complete
print("Monitoring eval runs...\n")
start_time = time.time()
check_interval = 30  # seconds

while True:
    status = check_status()
    elapsed = int(time.time() - start_time)
    
    print(f"[{elapsed//60:02d}:{elapsed%60:02d}] ", end="")
    print(f"✅ {status['completed']} | ", end="")
    print(f"⏳ {status['in_progress']} | ", end="")
    print(f"❌ {status['failed']}")
    
    if status['completed'] + status['failed'] == len(run_ids):
        print(f"\n🎉 All runs complete! Total time: {elapsed//60}m {elapsed%60}s")
        break
    
    time.sleep(check_interval)

Monitoring eval runs...

[00:06] ✅ 6 | ⏳ 0 | ❌ 0

🎉 All runs complete! Total time: 0m 6s


## 7. Fetch and Display Results

Retrieve detailed results for all models and display comparative analysis.

In [8]:
# Fetch detailed results
results = {}

for model_name, run_id in run_ids.items():
    url = f"{BASE_URL}/evals/{eval_id}/runs/{run_id}?api-version=2025-11-15-preview"
    response = requests.get(url, headers=get_headers())
    
    if response.status_code == 200:
        data = response.json()
        status = data.get('status')
        
        results[model_name] = {
            'status': status,
            'overall': data.get('result_counts', {}),
            'per_criteria': {},
            'report_url': data.get('report_url', '')
        }
        
        # Parse per-criteria results
        for c in data.get('per_testing_criteria_results', []):
            crit_name = c.get('testing_criteria', 'unknown')
            results[model_name]['per_criteria'][crit_name] = {
                'passed': c.get('passed', 0),
                'failed': c.get('failed', 0),
                'total': c.get('passed', 0) + c.get('failed', 0)
            }

print(f"✅ Results fetched for {len(results)} models")

✅ Results fetched for 6 models


## 8. Results Summary Table

In [9]:
print("="*110)
print("EVALUATION RESULTS - ALL MODELS")
print("="*110)
print()
print(f"{'Model':<15} {'Overall':<15} {'zava_quality':<18} {'IntentResolution':<18} {'TaskCompletion':<18}")
print("-"*100)

# Sort by zava_quality performance
sorted_models = sorted(results.keys(), key=lambda m: 
    results[m]['per_criteria'].get('zava_quality', {}).get('passed', 0) / 
    max(results[m]['per_criteria'].get('zava_quality', {}).get('total', 1), 1),
    reverse=True
)

for model_name in sorted_models:
    r = results[model_name]
    overall = r['overall']
    total = overall.get('total', 0)
    
    if total == 0:
        continue
    
    overall_str = f"{overall.get('passed', 0)}/{total}"
    overall_pct = overall.get('passed', 0) / total * 100
    
    # Get each criterion
    zava = r['per_criteria'].get('zava_quality', {'passed': 0, 'total': 0})
    zava_total = zava.get('total', 0)
    zava_str = f"{zava['passed']}/{zava_total} ({zava['passed']/zava_total*100:.0f}%)" if zava_total > 0 else "N/A"
    
    intent = r['per_criteria'].get('IntentResolution', {'passed': 0, 'total': 0})
    intent_total = intent.get('total', 0)
    intent_str = f"{intent['passed']}/{intent_total} ({intent['passed']/intent_total*100:.0f}%)" if intent_total > 0 else "N/A"
    
    task = r['per_criteria'].get('TaskCompletion', {'passed': 0, 'total': 0})
    task_total = task.get('total', 0)
    task_str = f"{task['passed']}/{task_total} ({task['passed']/task_total*100:.0f}%)" if task_total > 0 else "N/A"
    
    # Highlight best model
    prefix = "⭐" if model_name == sorted_models[0] else "  "
    
    print(f"{prefix} {model_name:<13} {overall_str:<7} ({overall_pct:.0f}%)  {zava_str:<18} {intent_str:<18} {task_str:<18}")

print("\n" + "="*100)
print("\nCriteria Definitions:")
print("  - zava_quality: Custom Python grader (Decision 50% + Financial 30% + Format 20%)")
print("  - IntentResolution: Built-in evaluator (understanding user intent)")
print("  - TaskCompletion: Built-in evaluator (completeness of response)")
print(f"\n⭐ BEST BASELINE: {sorted_models[0]}")
print("="*100)

EVALUATION RESULTS - ALL MODELS

Model           Overall         zava_quality       IntentResolution   TaskCompletion    
----------------------------------------------------------------------------------------------------
⭐ gpt-5-4       49/62   (79%)  49/62 (79%)        N/A                N/A               
   gpt-4-1       45/62   (73%)  45/57 (79%)        N/A                N/A               
   o4-mini       47/62   (76%)  47/62 (76%)        N/A                N/A               
   gpt-5-4-mini  41/62   (66%)  41/58 (71%)        N/A                N/A               
   gpt-4-1-mini  33/62   (53%)  33/62 (53%)        N/A                N/A               
   gpt-4-1-nano  27/62   (44%)  27/62 (44%)        N/A                N/A               


Criteria Definitions:
  - zava_quality: Custom Python grader (Decision 50% + Financial 30% + Format 20%)
  - IntentResolution: Built-in evaluator (understanding user intent)
  - TaskCompletion: Built-in evaluator (completeness of response)

⭐

## 9. Save Results

In [11]:
# Save run metadata
output_file = f"../eval/eval_results_{TIMESTAMP}.json"

output_data = {
    "timestamp": TIMESTAMP,
    "eval_id": eval_id,
    "eval_name": EVAL_NAME,
    "dataset_uri": DATASET_URI,
    "dataset_name": DATASET_NAME,
    "run_ids": run_ids,
    "results": results,
    "best_model": sorted_models[0] if sorted_models else None
}

with open(output_file, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"✅ Results saved to: {output_file}")
print(f"\nView detailed results in Azure AI Foundry:")
print(f"https://ai.azure.com/.../evaluations/{eval_id}")

✅ Results saved to: ../eval/eval_results_20260526-210056.json

View detailed results in Azure AI Foundry:
https://ai.azure.com/.../evaluations/eval_09201956538e459b8773c3502b7f257b
